In [1]:
# Paste this cell into your notebook
# Goal: pull (download) the latest registered MLflow model from Azure ML Model Registry to your local machine,
#       then load it as a Python object you can pass into RAIInsights.

import os
from dotenv import load_dotenv
from pathlib import Path
load_dotenv()

import mlflow
from mlflow.tracking import MlflowClient

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential


# -----------------------------
# 1) Your starter code (kept)
# -----------------------------
subscription_id = os.getenv("SUBSCRIPTION")
resource_group = os.getenv("RESOURCE_GROUP")
workspace_name = os.getenv("WS_NAME")

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)

workspace = ml_client.workspaces.get(ml_client.workspace_name)
tracking_uri = workspace.mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)

mlflow_client = MlflowClient(tracking_uri=tracking_uri)

print("✅ MLflow tracking URI set to:", tracking_uri)


c:\Users\micha\miniconda3\envs\admission_env\lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251
c:\Users\micha\miniconda3\envs\admission_env\lib\site-packages\pydantic\_internal\_config.py:383: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


✅ MLflow tracking URI set to: azureml://eastus.api.azureml.ms/mlflow/v1.0/subscriptions/a6b2ef6f-dbdd-4051-8a2e-9d1828d81dc2/resourceGroups/rg-admissions/providers/Microsoft.MachineLearningServices/workspaces/Admissions-Workspace


In [3]:
import sys
from pathlib import Path


In [5]:
try:
    parent_dir = Path(__file__).parent.parent
except NameError:
    parent_dir = Path.cwd().parent
sys.path.append(str(parent_dir))

In [6]:
from command_job import artifact_path_name

In [24]:
# Paste this cell into your notebook
# -----------------------------
# 2) Choose the registered model name (and optional version)
# -----------------------------
MODEL_NAME = artifact_path_name # <-- change this
MODEL_VERSION = None          # None = newest/latest version under this name

# (Optional) Where to download the model locally
DOWNLOAD_ROOT = Path("./downloaded_models").resolve()
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

print("Model name:", MODEL_NAME)
print("Download root:", DOWNLOAD_ROOT)


Model name: admissions_model
Download root: G:\My Drive\Admissions Model\register_model\downloaded_models


In [34]:
# Paste this cell into your notebook
# -----------------------------
# 3) Resolve the latest model version and download it locally
# -----------------------------
import os

# If no explicit version provided, find the latest registered version
if MODEL_VERSION is None:
    versions = list(ml_client.models.list(name=MODEL_NAME))
    if not versions:
        raise ValueError(f"No registered models found with name: {MODEL_NAME}")
    try:
        latest_model = max(versions, key=lambda m: int(m.version))
        print("3 worked")
    except Exception:
        latest_model = versions[-1]
else:
    latest_model = ml_client.models.get(name=MODEL_NAME, version=MODEL_VERSION)
    print("4 worked")

print("✅ Resolved model:")
print("   name   =", latest_model.name)
print("   version=", latest_model.version)

# Download the model artifacts to your machine
# This will create a folder like: ./downloaded_models/<name>/<version>/
local_model_dir = ml_client.models.download(
    name=latest_model.name,
    version=latest_model.version,
    download_path=str(DOWNLOAD_ROOT),
)

print("✅ Downloaded to:", local_model_dir)
print("Contents:", os.listdir(local_model_dir))


3 worked
✅ Resolved model:
   name   = admissions_model
   version= 12


✅ Downloaded to: None
Contents: ['register.py', 'reg_requirements.txt', 'pull_model.ipynb', 'downloaded_models']


# 

# Quick Other Download method using MLFlow

In [10]:
import mlflow.sklearn

MODEL_NAME = artifact_path_name

model_uri = f"models:/{MODEL_NAME}/latest"

model_obj = mlflow.sklearn.load_model(model_uri)

print("✅ Loaded sklearn model by registry reference")
print("Model type:", type(model_obj))


✅ Loaded sklearn model by registry reference
Model type: <class 'sklearn.pipeline.Pipeline'>
